<a href="https://colab.research.google.com/github/ankan-git-coder/Deep-learning-COURSE/blob/main/Lung_cancer_detection_using_densenet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [74]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets,transforms,models
from sklearn.metrics import classification_report, confusion_matrix

In [75]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohamedhanyyy/chest-ctscan-images")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'chest-ctscan-images' dataset.
Path to dataset files: /kaggle/input/chest-ctscan-images


In [76]:
DATA_DIR = path

In [78]:
TRAIN_DIR = os.path.join(DATA_DIR, "Data", "train")
VALID_DIR = os.path.join(DATA_DIR, "Data", "valid")
TEST_DIR = os.path.join(DATA_DIR, "Data", "test")

In [79]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using accelerator:{device}")

using accelerator:cuda


In [81]:
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 0.0005
NUM_CLASSES =4

In [82]:
train_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [83]:
val_test_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [84]:
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)
val_dataset = datasets.ImageFolder(root=VALID_DIR, transform=val_test_transforms)
test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=val_test_transforms)

In [85]:
train_dataset

Dataset ImageFolder
    Number of datapoints: 613
    Root location: /kaggle/input/chest-ctscan-images/Data/train
    StandardTransform
Transform: Compose(
               Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
               RandomHorizontalFlip(p=0.5)
               RandomRotation(degrees=[-10.0, 10.0], interpolation=nearest, expand=False, fill=0)
               ToTensor()
               Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
           )

In [86]:
train_loader = DataLoader(train_dataset,batch_size = BATCH_SIZE, shuffle=True,num_workers=2)
val_loader = DataLoader(val_dataset,batch_size = BATCH_SIZE, shuffle=False,num_workers=2)
test_loader = DataLoader(test_dataset,batch_size = BATCH_SIZE, shuffle=False,num_workers=2)

In [87]:
weights = models.DenseNet121_Weights.DEFAULT
model = models.densenet121(weights=weights)

In [89]:
for param in model.parameters():
    param.requires_grad = False

In [90]:
num_ftrs = model.classifier.in_features
model.classifier = nn.Sequential(
    nn.Linear(num_ftrs,256),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.4),
    nn.Linear(256,NUM_CLASSES)
)

In [91]:
model = model.to(device)

In [92]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(),lr = LEARNING_RATE)


In [93]:
print("\n=== training started===")
for epoch in range(EPOCHS):
    start_time = time.time()

    model.train()
    running_loss, correct_train, total_train = 0.0,0,0
    for images,labels in train_loader:
        images,labels = images.to(device),labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs,labels)
        loss.backward()
        optimizer.step()
        running_loss +=loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total_train += labels.size(0)
        correct_train += predicted.eq(labels).sum().item()
    model.eval()
    val_loss, correct_val, total_val = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total_val += labels.size(0)
            correct_val += predicted.eq(labels).sum().item()

    print(f"Epoch [{epoch+1}/{EPOCHS}] ({time.time()-start_time:.1f}s) -> "
          f"Train Loss: {running_loss/len(train_dataset):.4f} | Train Acc: {(correct_train/total_train)*100:.2f}% | "
          f"Val Loss: {val_loss/len(val_dataset):.4f} | Val Acc: {(correct_val/total_val)*100:.2f}%")




=== training started===
Epoch [1/20] (4.9s) -> Train Loss: 1.2628 | Train Acc: 42.09% | Val Loss: 1.2300 | Val Acc: 43.06%
Epoch [2/20] (5.1s) -> Train Loss: 1.0286 | Train Acc: 55.79% | Val Loss: 1.0577 | Val Acc: 52.78%
Epoch [3/20] (5.7s) -> Train Loss: 0.8875 | Train Acc: 61.83% | Val Loss: 0.9806 | Val Acc: 55.56%
Epoch [4/20] (4.8s) -> Train Loss: 0.8585 | Train Acc: 62.64% | Val Loss: 0.9496 | Val Acc: 55.56%
Epoch [5/20] (5.8s) -> Train Loss: 0.7730 | Train Acc: 69.00% | Val Loss: 0.9446 | Val Acc: 56.94%
Epoch [6/20] (5.0s) -> Train Loss: 0.7504 | Train Acc: 70.15% | Val Loss: 0.9172 | Val Acc: 58.33%
Epoch [7/20] (5.4s) -> Train Loss: 0.7741 | Train Acc: 67.21% | Val Loss: 0.8492 | Val Acc: 59.72%
Epoch [8/20] (5.3s) -> Train Loss: 0.7194 | Train Acc: 72.43% | Val Loss: 0.8158 | Val Acc: 62.50%
Epoch [9/20] (5.6s) -> Train Loss: 0.7125 | Train Acc: 72.10% | Val Loss: 0.8793 | Val Acc: 58.33%
Epoch [10/20] (6.1s) -> Train Loss: 0.6517 | Train Acc: 73.74% | Val Loss: 0.8403 | 

In [94]:
print('\n=== final test set evaluation===')
model.eval()
all_preds,all_labels = [],[]
with torch.no_grad():
    for images ,labels in test_loader:
        images = images.to(device)
        outputs= model(images)
        _,predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=test_dataset.classes))


=== final test set evaluation===
                         precision    recall  f1-score   support

         adenocarcinoma       0.60      0.81      0.69       120
   large.cell.carcinoma       0.58      0.61      0.60        51
                 normal       0.98      1.00      0.99        54
squamous.cell.carcinoma       0.71      0.36      0.47        90

               accuracy                           0.68       315
              macro avg       0.72      0.69      0.69       315
           weighted avg       0.69      0.68      0.66       315

